In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate momentum, efficiency, relative performance, and factor exposure for a single asset.
#Original Risk Analysis blocks included here: 14-21.


In [ ]:
# Block 2: import libraries and initialize analytics services
import logging
import warnings
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.visualization import (
    Plotter,
    )
from Quantapp.visualization.core import (
    configure_plotly_notebook_renderers,
    )
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency import (
    plot_benchmark_zscore_detail,
    plot_candlestick_drawdown_recovery_view,
    plot_momentum_zscore_comparison,
    plot_multi_benchmark_sharpe_spread_summary,
    plot_momentum_window_diagnostics_grid_view,
    plot_rolling_correlation_view,
    plot_seasonality_stack_view,
    plot_sharpe_sortino_comparison,
    plot_sharpe_surface_view,
    plot_sharpe_zscore_heatmap_view,
    plot_vix_fix_bands,
    )
from Quantapp.analytics import compute
from Quantapp.analytics import (
    Metric,
    SeriesTransforms,
    )
from Quantapp.data import get_market_history

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
metric = Metric()
series_transforms = SeriesTransforms()

def risk_adjusted_returns(data, windows, ratio_type='sharpe', risk_free_rate=0.0, annualization_factor=252):
    if isinstance(windows, (str, bytes)):
        raise ValueError("windows must be an integer or an iterable of integers")
    try:
        window_list = [int(window) for window in windows]
    except TypeError:
        window_list = [int(windows)]
    if not window_list or any(window <= 0 for window in window_list):
        raise ValueError("windows must contain positive integers")

    price_frame = data.to_frame(name=data.name or "price") if isinstance(data, pd.Series) else data
    if not isinstance(price_frame, pd.DataFrame):
        raise TypeError("data must be a pandas Series or DataFrame")

    returns = price_frame.pct_change()
    if isinstance(risk_free_rate, pd.Series):
        periodic_rate = risk_free_rate.astype(float).sort_index().reindex(returns.index).ffill()
    elif np.isscalar(risk_free_rate):
        periodic_rate = pd.Series((1.0 + float(risk_free_rate)) ** (1.0 / annualization_factor) - 1.0, index=returns.index)
    else:
        raise TypeError("risk_free_rate must be a scalar annual rate or a pandas Series")

    excess_returns = returns.sub(periodic_rate, axis=0)
    single_window = len(window_list) == 1
    single_series = price_frame.shape[1] == 1
    output = []
    for column in returns.columns:
        excess = excess_returns[column]
        for window in window_list:
            mean_excess = excess.rolling(window).mean()
            if ratio_type == 'sharpe':
                volatility = excess.rolling(window).std()
                ratio = np.sqrt(annualization_factor) * mean_excess / volatility
                ratio = ratio.where(volatility > 0)
            elif ratio_type == 'sortino':
                downside = excess.where(excess < 0, 0.0)
                downside_deviation = downside.rolling(window).apply(lambda values: np.sqrt((values**2).mean()), raw=True)
                ratio = np.sqrt(annualization_factor) * mean_excess / downside_deviation
            else:
                raise ValueError("Invalid ratio_type. Use 'sharpe' or 'sortino'.")

            ratio = ratio.replace([np.inf, -np.inf], np.nan)
            ratio.name = f"{ratio_type}_ratio_{window}" if single_window and single_series else f"{column}_{ratio_type}_{window}"
            output.append(ratio)
    return pd.concat(output, axis=1)


In [ ]:
# Block 3: initialize plotting helpers and Momentum & Efficiency display theme

import plotly.graph_objects as go

qp = Plotter()

# This is intentionally notebook-local: importing Quantapp.visualization does not apply this theme.
configure_plotly_notebook_renderers()

CURRENT_VALUE_REFERENCE_META_KEY = "quantapp_current_value_reference"
CURRENT_VALUE_REFERENCE_LINE_STYLE = dict(color="rgba(250, 204, 21, 0.82)", width=2, dash="dash")

def _is_current_value_reference_trace(trace):
    meta = getattr(trace, "meta", None)
    if isinstance(meta, dict) and meta.get(CURRENT_VALUE_REFERENCE_META_KEY):
        return True
    return str(getattr(trace, "name", "")).endswith(" Current Value")

def _current_value_reference_visibility(source_visible):
    return False if source_visible is False else "legendonly"

def _numeric_trace_y(trace):
    y_values = getattr(trace, "y", None)
    if y_values is None:
        return None, None
    numeric_y = pd.to_numeric(pd.Series(list(y_values)), errors="coerce")
    numeric_array = numeric_y.to_numpy(dtype=float)
    finite_mask = np.isfinite(numeric_array)
    return numeric_array, finite_mask

def _is_line_trace_for_current_value(trace):
    if _is_current_value_reference_trace(trace):
        return False
    if getattr(trace, "type", None) not in {"scatter", "scattergl"}:
        return False
    if "lines" not in str(getattr(trace, "mode", "")):
        return False
    if getattr(trace, "hoverinfo", None) == "skip":
        return False
    fill = getattr(trace, "fill", None)
    if fill not in (None, "none"):
        return False

    numeric_y, finite_mask = _numeric_trace_y(trace)
    if numeric_y is None or finite_mask.sum() < 2:
        return False
    return np.unique(numeric_y[finite_mask]).size > 1

def _trace_current_value_reference_payload(trace):
    numeric_y, finite_mask = _numeric_trace_y(trace)
    if numeric_y is None or finite_mask.sum() < 2:
        return None

    x_values = getattr(trace, "x", None)
    if x_values is None:
        x_values = list(range(len(numeric_y)))
    else:
        x_values = list(x_values)
    if len(x_values) != len(numeric_y):
        x_values = list(range(len(numeric_y)))

    valid_x = [x_value for x_value, is_valid in zip(x_values, finite_mask) if is_valid]
    current_value = numeric_y[finite_mask][-1]
    return valid_x[0], valid_x[-1], current_value

def _extend_visibility_buttons_for_current_value_lines(fig, source_indices):
    if not source_indices:
        return
    original_trace_count = len(fig.data) - len(source_indices)
    for menu in fig.layout.updatemenus or []:
        for button in menu.buttons or []:
            args = list(button.args or [])
            if not args or not isinstance(args[0], dict) or "visible" not in args[0]:
                continue
            visible = list(args[0]["visible"])
            if len(visible) != original_trace_count:
                continue
            args[0]["visible"] = visible + [
                _current_value_reference_visibility(visible[source_index])
                for source_index in source_indices
            ]
            button.args = tuple(args)

def _style_current_value_reference_lines(fig):
    for trace in fig.data:
        if _is_current_value_reference_trace(trace):
            trace.update(line=CURRENT_VALUE_REFERENCE_LINE_STYLE.copy(), visible="legendonly", showlegend=True)

def _add_current_value_reference_lines(fig):
    if any(_is_current_value_reference_trace(trace) for trace in fig.data):
        _style_current_value_reference_lines(fig)
        return fig

    source_indices = []
    for source_index, trace in enumerate(list(fig.data)):
        if not _is_line_trace_for_current_value(trace):
            continue
        payload = _trace_current_value_reference_payload(trace)
        if payload is None:
            continue
        x_start, x_end, current_value = payload
        reference_trace = dict(
            x=[x_start, x_end],
            y=[current_value, current_value],
            mode="lines",
            name=f"{getattr(trace, 'name', '') or 'Series'} Current Value",
            line=CURRENT_VALUE_REFERENCE_LINE_STYLE.copy(),
            hoverinfo="skip",
            showlegend=True,
            visible=_current_value_reference_visibility(getattr(trace, "visible", True)),
            meta={CURRENT_VALUE_REFERENCE_META_KEY: True},
        )
        xaxis = getattr(trace, "xaxis", None)
        yaxis = getattr(trace, "yaxis", None)
        if xaxis:
            reference_trace["xaxis"] = xaxis
        if yaxis:
            reference_trace["yaxis"] = yaxis
        fig.add_trace(go.Scatter(**reference_trace))
        source_indices.append(source_index)

    _extend_visibility_buttons_for_current_value_lines(fig, source_indices)
    return fig

if not hasattr(go.Figure, "_quantapp_original_show"):
    go.Figure._quantapp_original_show = go.Figure.show

def _quantapp_show_with_current_value_lines(self, *args, **kwargs):
    _add_current_value_reference_lines(self)
    return go.Figure._quantapp_original_show(self, *args, **kwargs)

go.Figure.show = _quantapp_show_with_current_value_lines

In [ ]:
# Block 4: set notebook parameters

ticker_str = "EFA"
vix_str = "^VIX"
interval = "1d"
period = "20y"
risk_free_ticker = "^IRX"
benchmark_tickers = ["SPY"]
time_frame_map = {"short": 21, "mid": 50, "long": 200}
selected_time_frames = [21, 50, 200]
default_window = 200
length_of_plots = 20
var_position_value = None

In [ ]:
# Block 5: fetch market history and assign notebook roles
requested_symbols = [
    ticker_str,
    vix_str,
    risk_free_ticker,
    *benchmark_tickers,
]

asset_histories = get_market_history(
    symbols=requested_symbols,
    period=period,
    interval=interval,
    provider="yfinance",
    align=True,
)

asset_history           = asset_histories.get(ticker_str, pd.DataFrame())
vix_history             = asset_histories.get(vix_str, pd.DataFrame())
risk_free_proxy_history = asset_histories.get(risk_free_ticker, pd.DataFrame())

benchmark_data = {
    symbol: frame
    for symbol, frame in asset_histories.items()
    if symbol not in {ticker_str, vix_str, risk_free_ticker}
}

#loaded_benchmark_tickers = list(benchmark_data)
#analysis_index = asset_history.index


In [ ]:
# Block 7: derive analysis series from normalized market data

risk_free_daily_rate = series_transforms.annualized_yield_to_periodic_rate(
    risk_free_proxy_history,
    annualization_factor=252,
    input_is_percent=True,
    lag_periods=1,
    reference_index=asset_history.index,
)

ticker_monthly_data = series_transforms.resample(asset_history, frequency="monthly")
ticker_weekly_data = series_transforms.resample(asset_history, frequency="weekly")
ticker_daily_data = series_transforms.resample(asset_history, frequency="daily")

ticker_monthly_returns = ticker_monthly_data["Close"].pct_change(fill_method=None).dropna()
ticker_weekly_returns = ticker_weekly_data["Close"].pct_change(fill_method=None).dropna()
ticker_daily_returns = ticker_daily_data["Close"].pct_change(fill_method=None).dropna()


In [ ]:
# Block 8: compute rolling arithmetic/geometric mean returns with MAD-score details

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency._shared import (
    finalize_dark_figure,
    header_margin,
    header_title,
)

configured_rolling_mean_windows = globals().get(
    "selected_time_frames",
    [globals().get("default_window", 200)],
)
if isinstance(configured_rolling_mean_windows, (int, float, np.integer)):
    configured_rolling_mean_windows = [configured_rolling_mean_windows]

rolling_mean_windows = []
for window in configured_rolling_mean_windows:
    try:
        window = int(window)
    except (TypeError, ValueError):
        continue
    if window > 0 and window not in rolling_mean_windows:
        rolling_mean_windows.append(window)
if not rolling_mean_windows:
    rolling_mean_windows = [200]

preferred_rolling_mean_window = int(globals().get("default_window", 200))
rolling_mean_window = (
    preferred_rolling_mean_window
    if preferred_rolling_mean_window in rolling_mean_windows
    else 200
    if 200 in rolling_mean_windows
    else max(rolling_mean_windows)
)

def rolling_arithmetic_label(window):
    return f"{int(window)}D Arithmetic Mean"

def rolling_geometric_label(window):
    return f"{int(window)}D Geometric Mean"

compounding_efficiency_label = "Compounding Efficiency"
volatility_drag_label = "Volatility Drag"

def compounding_efficiency_from_means(arithmetic_mean, geometric_mean):
    return (1.0 + geometric_mean).div(1.0 + arithmetic_mean).replace([np.inf, -np.inf], np.nan)

def volatility_drag_from_means(arithmetic_mean, geometric_mean):
    return arithmetic_mean - geometric_mean

def rolling_metric_mad_score(series):
    clean = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    median = clean.median()
    mad = (clean - median).abs().median()
    if mad == 0 or pd.isna(mad):
        return pd.Series(0.0, index=clean.index)
    return ((clean - median) / (1.4826 * mad)).dropna()

def metric_detail_array(value_series, mad_score_series, index):
    detail_frame = pd.concat(
        {
            "value": pd.Series(value_series),
            "mad_score": pd.Series(mad_score_series),
        },
        axis=1,
    ).reindex(index)
    return detail_frame[["value", "mad_score"]].to_numpy()

def mad_score_annotation_text(metric_label, metric_symbol, latex_formula, function_name):
    return (
        f"<b>{metric_label}</b><br>"
        f"${metric_symbol}_t = {latex_formula}$<br>"
        rf"$m_t = \frac{{{metric_symbol}_t - \operatorname{{median}}({metric_symbol})}}{{1.4826 \cdot \operatorname{{MAD}}({metric_symbol})}}$<br>"
        rf"$\operatorname{{MAD}}({metric_symbol}) = \operatorname{{median}}(|{metric_symbol} - \operatorname{{median}}({metric_symbol})|)$<br>"
        f"<span style=\"font-family:monospace;font-size:10px\">{function_name}</span>"
    )

def add_formula_annotation(fig, text, y_position):
    fig.add_annotation(
        text=text,
        x=0.01,
        y=y_position,
        xref="paper",
        yref="paper",
        xanchor="left",
        yanchor="top",
        showarrow=False,
        align="left",
        bgcolor="rgba(15, 23, 42, 0.72)",
        bordercolor="rgba(148, 163, 184, 0.34)",
        borderwidth=1,
        font=dict(size=11, color="rgba(226, 232, 240, 0.95)"),
    )

def block8_dropdown_menu(buttons, x, active=0):
    return dict(
        type="dropdown",
        buttons=buttons,
        direction="down",
        showactive=True,
        active=active,
        x=x,
        xanchor="left",
        y=1.09,
        yanchor="top",
        bgcolor="rgba(15, 23, 42, 0.92)",
        bordercolor="rgba(148, 163, 184, 0.35)",
        font=dict(color="rgba(226, 232, 240, 0.96)"),
    )

def block8_time_range_buttons(global_start, global_end, axis_count):
    def make_range(years=None):
        start = global_start if years is None else max(global_start, global_end - pd.DateOffset(years=years))
        return {
            ("xaxis.range" if axis_idx == 1 else f"xaxis{axis_idx}.range"): [start, global_end]
            for axis_idx in range(1, axis_count + 1)
        }

    return [
        dict(label="10 Years", method="relayout", args=[make_range(10)]),
        dict(label="5 Years", method="relayout", args=[make_range(5)]),
        dict(label="3 Years", method="relayout", args=[make_range(3)]),
        dict(label="1 Year", method="relayout", args=[make_range(1)]),
        dict(label="All", method="relayout", args=[make_range(None)]),
    ]

rolling_mean_returns = ticker_daily_returns.dropna().sort_index()

def rolling_mean_metric_frames(daily_returns, window):
    window = int(window)
    returns = pd.Series(daily_returns).dropna().sort_index()
    arithmetic_mean = returns.rolling(
        window,
        min_periods=window,
    ).mean()
    geometric_mean = np.expm1(
        np.log1p(returns).rolling(
            window,
            min_periods=window,
        ).mean()
    )
    compounding_efficiency = compounding_efficiency_from_means(arithmetic_mean, geometric_mean)
    volatility_drag = volatility_drag_from_means(arithmetic_mean, geometric_mean)
    arithmetic_label = rolling_arithmetic_label(window)
    geometric_label = rolling_geometric_label(window)

    values = pd.concat(
        {
            arithmetic_label: arithmetic_mean,
            geometric_label: geometric_mean,
            compounding_efficiency_label: compounding_efficiency,
            volatility_drag_label: volatility_drag,
        },
        axis=1,
    ).dropna(how="all")
    mad_scores = pd.concat(
        {
            arithmetic_label: rolling_metric_mad_score(arithmetic_mean),
            geometric_label: rolling_metric_mad_score(geometric_mean),
            compounding_efficiency_label: rolling_metric_mad_score(compounding_efficiency),
            volatility_drag_label: rolling_metric_mad_score(volatility_drag),
        },
        axis=1,
    ).dropna(how="all")
    return values, mad_scores

asset_rolling_mean_detail = {}
for window in rolling_mean_windows:
    values, mad_scores = rolling_mean_metric_frames(rolling_mean_returns, window)
    if not values.empty:
        asset_rolling_mean_detail[window] = {
            "values": values,
            "mad_scores": mad_scores,
        }
if not asset_rolling_mean_detail:
    raise ValueError("No rolling mean data available for Block 8.")
if rolling_mean_window not in asset_rolling_mean_detail:
    rolling_mean_window = next(iter(asset_rolling_mean_detail))

rolling_mean_comparison = asset_rolling_mean_detail[rolling_mean_window]["values"]
rolling_mean_mad_comparison = asset_rolling_mean_detail[rolling_mean_window]["mad_scores"]

benchmark_rolling_mean_detail = {window: {} for window in asset_rolling_mean_detail}
for benchmark_symbol, benchmark_frame in benchmark_data.items():
    if isinstance(benchmark_frame, pd.DataFrame):
        if "Close" not in benchmark_frame:
            continue
        benchmark_close = benchmark_frame["Close"]
    else:
        benchmark_close = pd.Series(benchmark_frame)

    benchmark_returns = benchmark_close.pct_change(fill_method=None)
    for window in asset_rolling_mean_detail:
        benchmark_values, benchmark_mad_scores = rolling_mean_metric_frames(benchmark_returns, window)
        if benchmark_values.empty:
            continue
        benchmark_rolling_mean_detail[window][benchmark_symbol] = {
            "values": benchmark_values,
            "mad_scores": benchmark_mad_scores,
        }

benchmark_overlay_order = []
for window_payload in benchmark_rolling_mean_detail.values():
    for benchmark_symbol in window_payload:
        if benchmark_symbol not in benchmark_overlay_order:
            benchmark_overlay_order.append(benchmark_symbol)
benchmark_line_dashes = ["dot", "dash", "longdash", "dashdot"]
benchmark_line_dash_map = {
    symbol: benchmark_line_dashes[index % len(benchmark_line_dashes)]
    for index, symbol in enumerate(benchmark_overlay_order)
}

rolling_mean_subplot_count = 3

fig_rolling_mean_comparison = make_subplots(
    rows=rolling_mean_subplot_count,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.36, 0.32, 0.32],
    subplot_titles=(
        "Rolling Mean Return MAD Scores",
        f"{compounding_efficiency_label} MAD Score",
        f"{volatility_drag_label} MAD Score",
    ),
)

def block8_plot_title(window):
    return f"{ticker_str}: {int(window)}-Day Rolling Mean, Compounding Efficiency, and Volatility Drag MAD Scores vs Benchmarks"

def metric_trace_specs(window):
    return [
        (rolling_arithmetic_label(window), 1, "#38BDF8", "Mean Return", ".2%", True),
        (rolling_geometric_label(window), 1, "#F97316", "Mean Return", ".2%", True),
        (compounding_efficiency_label, 2, "#A3E635", "Efficiency", ".4f", True),
        (volatility_drag_label, 3, "#E879F9", "Volatility Drag", ".2%", True),
    ]

def add_metric_trace(
    name_prefix,
    values,
    mad_scores,
    metric_label,
    row,
    color,
    raw_label,
    raw_format,
    line_dash="solid",
    line_width=2.0,
    opacity=1.0,
    visible=True,
):
    series = mad_scores.get(metric_label, pd.Series(dtype=float)).dropna()
    if series.empty:
        return

    raw_value_template = "%{customdata[0]:" + raw_format + "}"
    hovertemplate = (
        "%{x|%Y-%m-%d}<br>"
        "MAD Score: %{y:.2f}<br>"
        f"{raw_label}: " + raw_value_template + "<extra></extra>"
    )
    fig_rolling_mean_comparison.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode="lines",
            name=f"{name_prefix} {metric_label} MAD Score",
            line=dict(color=color, width=line_width, dash=line_dash),
            opacity=opacity,
            visible=visible,
            customdata=metric_detail_array(
                values.get(metric_label, pd.Series(dtype=float)),
                mad_scores.get(metric_label, pd.Series(dtype=float)),
                series.index,
            ),
            hovertemplate=hovertemplate,
        ),
        row=row,
        col=1,
    )

window_trace_indices = {}
for window, asset_payload in asset_rolling_mean_detail.items():
    window_visible = window == rolling_mean_window
    trace_start = len(fig_rolling_mean_comparison.data)
    for metric_label, row, color, raw_label, raw_format, _ in metric_trace_specs(window):
        add_metric_trace(
            name_prefix=ticker_str,
            values=asset_payload["values"],
            mad_scores=asset_payload["mad_scores"],
            metric_label=metric_label,
            row=row,
            color=color,
            raw_label=raw_label,
            raw_format=raw_format,
            visible=window_visible,
        )

    for benchmark_symbol, benchmark_payload in benchmark_rolling_mean_detail.get(window, {}).items():
        for metric_label, row, color, raw_label, raw_format, _ in metric_trace_specs(window):
            add_metric_trace(
                name_prefix=benchmark_symbol,
                values=benchmark_payload["values"],
                mad_scores=benchmark_payload["mad_scores"],
                metric_label=metric_label,
                row=row,
                color=color,
                raw_label=raw_label,
                raw_format=raw_format,
                line_dash=benchmark_line_dash_map.get(benchmark_symbol, "dot"),
                line_width=1.4,
                opacity=0.82,
                visible=window_visible,
            )
    window_trace_indices[window] = list(range(trace_start, len(fig_rolling_mean_comparison.data)))

total_trace_count = len(fig_rolling_mean_comparison.data)
base_plot_title = block8_plot_title(rolling_mean_window)

def rolling_window_visibility(window):
    active_indices = set(window_trace_indices.get(window, []))
    return [trace_index in active_indices for trace_index in range(total_trace_count)]

rolling_window_order = list(asset_rolling_mean_detail)
rolling_window_dropdown_buttons = [
    dict(
        label=f"{int(window)} Days",
        method="update",
        args=[
            {"visible": rolling_window_visibility(window)},
            {"title": header_title(block8_plot_title(window))},
        ],
    )
    for window in rolling_window_order
]
rolling_window_dropdown_active = rolling_window_order.index(rolling_mean_window)

plot_x_indexes = [payload["mad_scores"].index for payload in asset_rolling_mean_detail.values()]
for window_payload in benchmark_rolling_mean_detail.values():
    plot_x_indexes.extend(
        payload["mad_scores"].index
        for payload in window_payload.values()
        if not payload["mad_scores"].empty
    )
plot_x_indexes = [index for index in plot_x_indexes if len(index) > 0]
plot_start = min((index.min() for index in plot_x_indexes), default=None)
plot_end = max((index.max() for index in plot_x_indexes), default=None)

block8_updatemenus = [
    block8_dropdown_menu(
        rolling_window_dropdown_buttons,
        x=0.0,
        active=rolling_window_dropdown_active,
    )
]
if plot_start is not None and plot_end is not None:
    default_start = max(plot_start, plot_end - pd.DateOffset(years=10))
    for subplot_row in range(1, rolling_mean_subplot_count + 1):
        fig_rolling_mean_comparison.update_xaxes(range=[default_start, plot_end], row=subplot_row, col=1)
    block8_updatemenus.append(
        block8_dropdown_menu(
            block8_time_range_buttons(plot_start, plot_end, rolling_mean_subplot_count),
            x=0.24,
            active=0,
        )
    )

for score_row in (1, 2, 3):
    lower_zone_color = "rgba(34, 197, 94, 0.16)" if score_row == 2 else "rgba(239, 68, 68, 0.16)"
    upper_zone_color = "rgba(239, 68, 68, 0.16)" if score_row == 2 else "rgba(34, 197, 94, 0.16)"
    neutral_lower_bound = -0.5 if score_row == 3 else -1
    neutral_upper_bound = 0.5 if score_row == 2 else 1
    lower_zone_bounds = (-1, -0.5) if score_row == 3 else (-2, -1)
    upper_zone_bounds = (0.5, 1) if score_row == 2 else (1, 2)
    positive_reference_levels = (0.5, 1) if score_row == 2 else (1, 2)
    negative_reference_levels = (0.5, 1) if score_row == 3 else (1, 2)
    fig_rolling_mean_comparison.add_hrect(
        y0=neutral_lower_bound,
        y1=neutral_upper_bound,
        fillcolor="rgba(148, 163, 184, 0.12)",
        line_width=0,
        layer="below",
        row=score_row,
        col=1,
    )
    fig_rolling_mean_comparison.add_hrect(
        y0=lower_zone_bounds[0],
        y1=lower_zone_bounds[1],
        fillcolor=lower_zone_color,
        line_width=0,
        layer="below",
        row=score_row,
        col=1,
    )
    fig_rolling_mean_comparison.add_hrect(
        y0=upper_zone_bounds[0],
        y1=upper_zone_bounds[1],
        fillcolor=upper_zone_color,
        line_width=0,
        layer="below",
        row=score_row,
        col=1,
    )
    fig_rolling_mean_comparison.add_hline(
        y=0,
        line_dash="solid",
        line_color="rgba(226, 232, 240, 0.70)",
        row=score_row,
        col=1,
    )
    for sigma_level in positive_reference_levels:
        fig_rolling_mean_comparison.add_hline(
            y=sigma_level,
            line_dash="dash",
            line_color="rgba(148, 163, 184, 0.55)",
            row=score_row,
            col=1,
        )
    for sigma_level in negative_reference_levels:
        fig_rolling_mean_comparison.add_hline(
            y=-sigma_level,
            line_dash="dash",
            line_color="rgba(148, 163, 184, 0.55)",
            row=score_row,
            col=1,
        )

score_zone_annotations = [
    (1, "Strong Mean Returns", 1.5, "rgba(235, 255, 235, 0.96)"),
    (1, "Weak Mean Returns", -1.5, "rgba(255, 235, 235, 0.96)"),
    (2, "Efficient Compounding", 0.75, "rgba(255, 235, 235, 0.96)"),
    (2, "Poor Compounding", -1.5, "rgba(235, 255, 235, 0.96)"),
    (3, "High Drag", 1.5, "rgba(235, 255, 235, 0.96)"),
    (3, "Low Drag", -0.75, "rgba(255, 235, 235, 0.96)"),
]

for zone_row, zone_label, zone_y, zone_color in score_zone_annotations:
    fig_rolling_mean_comparison.add_annotation(
        x=0.5,
        y=zone_y,
        xref="x domain",
        yref="y",
        text=zone_label,
        showarrow=False,
        xanchor="center",
        yanchor="middle",
        font=dict(color=zone_color, size=14),
        row=zone_row,
        col=1,
    )

add_formula_annotation(
    fig_rolling_mean_comparison,
    mad_score_annotation_text(
        compounding_efficiency_label,
        "CE",
        r"\frac{1 + g_t}{1 + a_t}",
        "compounding_efficiency_from_means(arithmetic_mean, geometric_mean)",
    ),
    y_position=0.61,
)
add_formula_annotation(
    fig_rolling_mean_comparison,
    mad_score_annotation_text(
        volatility_drag_label,
        "VD",
        r"a_t - g_t",
        "volatility_drag_from_means(arithmetic_mean, geometric_mean)",
    ),
    y_position=0.28,
)

fig_rolling_mean_comparison.update_layout(
    title=header_title(base_plot_title),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=header_margin(top=165),
    updatemenus=block8_updatemenus,
    height=1125,
)
fig_rolling_mean_comparison.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-5, 5], row=1, col=1)
fig_rolling_mean_comparison.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-6, 2], row=2, col=1)
fig_rolling_mean_comparison.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-2, 6], row=3, col=1)
fig_rolling_mean_comparison.update_xaxes(title_text="Date", row=3, col=1)
fig_rolling_mean_comparison = finalize_dark_figure(fig_rolling_mean_comparison)
fig_rolling_mean_comparison.show()


In [ ]:
# Block 9: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX

ticker_vix_fix = compute.rolling(asset_history['Close'], metric=metric.vix_fix, window=22)

fig = plot_vix_fix_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3],
)
fig.show()



In [ ]:
# Block 10: stack candlestick, drawdown comparison, and rolling recovery time
# Change selected_time_frames in Block 4 to a list like [21, 50, 200], then rerun this cell.

drawdown_recovery_by_window = {}

for window in selected_time_frames:
    
    rolling_peak = compute.rolling(
        asset_history['Close'],
        metric=pd.Series.max,
        window=window,
        min_periods=1,
    )
    
    underwater_series = asset_history['Close'].div(rolling_peak).sub(1.0).dropna()
    
    drawdown_series = compute.rolling(
        asset_history['Close'],
        metric=metric.textbook_window_drawdown,
        window=window,
        dropna=False,
    ).dropna()
    
    recovery_series = compute.rolling(
        asset_history['Close'],
        metric=metric.window_recovery_time,
        window=window,
        dropna=False,
    ).dropna()

    drawdown_recovery_by_window[window] = {
        'underwater': underwater_series,
        'max_drawdown': drawdown_series,
        'recovery_time': recovery_series,
    }

fig = plot_candlestick_drawdown_recovery_view(
    price_frame=asset_history,
    drawdown_recovery_by_window=drawdown_recovery_by_window,
    ticker_label=ticker_str,
    candlestick_period=period,
    default_timeframe_label='10 Years',
)
fig.show()


In [ ]:
# Block 11: compute rolling Sharpe windows, momentum histograms, and volatility

window_sizes = list(range(3, 201))
annualization_factor = 252

momentum_close = asset_history['Close'].dropna()

sharpe_table = risk_adjusted_returns(
    momentum_close,
    windows=window_sizes,
    ratio_type="sharpe",
    risk_free_rate=risk_free_daily_rate,
    annualization_factor=annualization_factor,
).set_axis(window_sizes, axis=1).replace([np.inf, -np.inf], np.nan).dropna(how="all").copy()

returns = momentum_close.pct_change()
excess_returns = returns - risk_free_daily_rate
volatility_df = np.sqrt(annualization_factor) * compute.rolling_windows(
    excess_returns,
    metric=pd.Series.std,
    windows=window_sizes,
)

momentum_diagnostics_context = {
    "sharpe_table": sharpe_table,
    "volatility_df": volatility_df,
}

fig_momentum_window_diagnostics_grid = plot_momentum_window_diagnostics_grid_view(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)
fig_sharpe_surface = plot_sharpe_surface_view(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

fig_momentum_window_diagnostics_grid.show()
fig_sharpe_surface.show()


In [ ]:
# Block 12: plot stacked rolling Sharpe z-score heatmaps for 1-200 day windows plus cross-window summaries

heatmap_windows = list(range(1, 201))

def heatmap_zscore(series):
    clean = pd.Series(series).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    std = clean.std()
    if std == 0 or pd.isna(std):
        return pd.Series(0.0, index=clean.index)
    return (clean - clean.mean()) / std

def rolling_sharpe_frame(close):
    return risk_adjusted_returns(
        close.dropna().sort_index(),
        windows=heatmap_windows,
        ratio_type="sharpe",
        risk_free_rate=risk_free_daily_rate,
    ).set_axis(heatmap_windows, axis=1)

asset_sharpe_frame = rolling_sharpe_frame(asset_history["Close"])

asset_sharpe_zscore_frame = asset_sharpe_frame.apply(heatmap_zscore)

benchmark_spread_zscore_frames = {}
for symbol, benchmark_frame in benchmark_data.items():
    benchmark_sharpe_frame = rolling_sharpe_frame(benchmark_frame["Close"])
    benchmark_spread = benchmark_sharpe_frame - asset_sharpe_frame
    benchmark_spread_zscore_frames[symbol] = benchmark_spread.apply(heatmap_zscore)


display(benchmark_spread_zscore_frames)
fig = plot_sharpe_zscore_heatmap_view(
    asset_sharpe_zscore_frame=asset_sharpe_zscore_frame,
    benchmark_spread_zscore_frames=benchmark_spread_zscore_frames,
    ticker_label=ticker_str,
)
fig.show()


In [ ]:
# Block 13: visualize monthly, weekly, and daily seasonality patterns

fig_ticker_seasonality_stack = plot_seasonality_stack_view(
    monthly_returns=ticker_monthly_returns,
    weekly_returns=ticker_weekly_returns,
    daily_returns=ticker_daily_returns,
    ticker_label=ticker_str,
    as_of=asset_history.index.max(),
)
fig_ticker_seasonality_stack.show()

In [ ]:
# Block 14: compute Sharpe/Sortino ratios and spreads

from Quantapp.analytics.series_utils import calculate_zscore

asset_close = asset_history['Close'].dropna().sort_index()
annualization_factor = 252
risk_time_frame_map = {str(term): int(window) for term, window in time_frame_map.items()}
selected_windows = sorted(dict.fromkeys(int(window) for window in selected_time_frames))
selected_time_frame_map = {}

for window in selected_windows:
    term_key = next(
        (term for term, mapped_window in risk_time_frame_map.items() if mapped_window == window),
        f"selected_{window}",
    )
    selected_time_frame_map[term_key] = window

risk_time_frame_map.update(selected_time_frame_map)

def rolling_ratio_series(close, window, ratio_type):
    ratio_frame = risk_adjusted_returns(
        close.dropna().sort_index(),
        windows=[window],
        ratio_type=ratio_type,
        risk_free_rate=risk_free_daily_rate,
    )
    return ratio_frame.iloc[:, 0]

def rolling_risk_components(close, window):
    close = close.dropna().sort_index()
    if isinstance(risk_free_daily_rate, pd.Series):
        periodic_risk_free_rate = risk_free_daily_rate.astype(float).sort_index().reindex(close.index).ffill()
    else:
        periodic_risk_free_rate = (1.0 + float(risk_free_daily_rate)) ** (1.0 / annualization_factor) - 1.0
    excess_returns = close.pct_change() - periodic_risk_free_rate
    rolling_mean = excess_returns.rolling(window).mean()
    rolling_std = excess_returns.rolling(window).std()
    sharpe_ratio = np.sqrt(annualization_factor) * rolling_mean / rolling_std
    return {
        "annualized_excess_return": annualization_factor * rolling_mean,
        "annualized_volatility": np.sqrt(annualization_factor) * rolling_std,
        "sharpe_ratio": sharpe_ratio.where(rolling_std > 0).replace([np.inf, -np.inf], np.nan),
    }

def zscore_or_empty(series):
    clean = pd.Series(series).dropna()
    return calculate_zscore(clean).dropna() if not clean.empty else pd.Series(dtype=float)

asset_sharpe_map = {}
asset_sortino_map = {}
asset_component_map = {}

for term, window in risk_time_frame_map.items():
    asset_sharpe_map[term] = rolling_ratio_series(asset_close, window, "sharpe")
    asset_sortino_map[term] = rolling_ratio_series(asset_close, window, "sortino")
    asset_component_map[term] = rolling_risk_components(asset_close, window)

asset_sharpe_sortino_spread_map = {
    term: asset_sortino_map[term] - asset_sharpe_map[term]
    for term in risk_time_frame_map
}

benchmark_metrics = {}
for symbol, benchmark_frame in benchmark_data.items():
    benchmark_close = benchmark_frame["Close"] if isinstance(benchmark_frame, pd.DataFrame) else benchmark_frame
    benchmark_close = benchmark_close.dropna().sort_index()
    benchmark_metrics[symbol] = {}

    for term, window in risk_time_frame_map.items():
        benchmark_components = rolling_risk_components(benchmark_close, window)
        benchmark_sharpe = benchmark_components["sharpe_ratio"]
        benchmark_metrics[symbol][term] = {
            "spread": benchmark_close.pct_change(window) - asset_close.pct_change(window),
            "annualized_excess_return": benchmark_components["annualized_excess_return"],
            "annualized_volatility": benchmark_components["annualized_volatility"],
            "sharpe_ratio": benchmark_sharpe,
            "sharpe_spread": benchmark_sharpe - asset_sharpe_map[term],
        }

benchmark_order = list(benchmark_metrics)
default_benchmark = benchmark_order[0] if benchmark_order else None
spread_plot_data = {
    term: {symbol: benchmark_metrics[symbol][term]["sharpe_spread"] for symbol in benchmark_order}
    for term in risk_time_frame_map
}

term_config_map = {}
for term, window in risk_time_frame_map.items():
    label = f"{window}-day"
    sharpe = asset_sharpe_map[term]
    sortino = asset_sortino_map[term]
    spread = asset_sharpe_sortino_spread_map[term]
    term_config_map[label] = {
        "sharpe": sharpe,
        "sortino": sortino,
        "spread": spread,
        "sharpe_zscore": zscore_or_empty(sharpe),
        "sortino_zscore": zscore_or_empty(sortino),
        "spread_zscore": zscore_or_empty(spread),
        "time_frame": window,
        "term_key": term,
    }

selected_term_config_map = {
    f"{window}-day": term_config_map[f"{window}-day"]
    for window in selected_time_frame_map.values()
    if f"{window}-day" in term_config_map
}


In [ ]:
# Block 15: plot rolling correlation of the asset versus benchmarks

asset_daily_returns = asset_history['Close'].pct_change(fill_method=None)
correlation_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
correlation_benchmark_order = benchmark_order if benchmark_order else list(benchmark_data.keys())
rolling_correlation_map = {}

for term in correlation_term_order:
    window = int(time_frame_map[term])
    term_series_map = {}

    for symbol in correlation_benchmark_order:
        benchmark_frame = benchmark_data.get(symbol)
        if benchmark_frame is None or 'Close' not in benchmark_frame:
            continue

        benchmark_daily_returns = benchmark_frame['Close'].pct_change(fill_method=None)
        aligned_returns = pd.concat(
            [
                asset_daily_returns.rename('asset'),
                benchmark_daily_returns.rename(symbol),
            ],
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns['asset'].rolling(window).corr(aligned_returns[symbol]).dropna()
        if rolling_correlation_series.empty:
            continue

        term_series_map[symbol] = rolling_correlation_series

    if term_series_map:
        rolling_correlation_map[term] = term_series_map

rolling_correlation_fig = plot_rolling_correlation_view(
    rolling_correlation_map=rolling_correlation_map,
    time_frame_map=time_frame_map,
    term_order=correlation_term_order,
    benchmark_order=correlation_benchmark_order,
    ticker_label=ticker_str,
)
rolling_correlation_fig.show()


In [ ]:
# Block 16: plot Sharpe & Sortino efficiency for the selected timeframe set
# Change selected_time_frames in Block 4 to a list like [21, 50, 200], then rerun the notebook.

fig = plot_sharpe_sortino_comparison(
    term_config_map=selected_term_config_map,
    ticker_label=ticker_str,
)
fig.show()


In [ ]:
# Block 17: render interactive momentum z-score comparisons

def momentum_zscore_map(close_series, window_pairs, normalizer_window=None, ddof=0):
    close = close_series.dropna()
    returns = close.pct_change()
    zscore_data = {}

    for label, pair in window_pairs.items():
        short_window, long_window = int(pair[0]), int(pair[1])
        short_average_return = compute.rolling_windows(returns, metric="mean", windows=[short_window])[short_window] * 100.0
        long_average_return = compute.rolling_windows(returns, metric="mean", windows=[long_window])[long_window] * 100.0
        momentum_diff = short_average_return - long_average_return

        if normalizer_window is None:
            mean = compute.latest(momentum_diff, metric=pd.Series.mean)
            std = compute.latest(momentum_diff, metric=pd.Series.std, ddof=ddof)
        else:
            normalizer_window = int(normalizer_window)
            mean = compute.rolling_windows(momentum_diff, metric="mean", windows=[normalizer_window])[normalizer_window]
            std = compute.rolling_windows(momentum_diff, metric="std", windows=[normalizer_window], ddof=ddof)[normalizer_window]

        if np.isscalar(std):
            std = np.nan if std == 0 else std
        else:
            std = std.replace(0, np.nan)

        zscore_data[str(label)] = (momentum_diff - mean) / std

    return zscore_data


window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_zscore_map(
    asset_history['Close'],
    window_pairs=window_pairs,
)

fig = plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="200 vs 400",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.update_layout(height=850)
fig.show()


In [ ]:
# Block 18: combine risk-adjusted return and benchmark plots
# Requires the current kernel session to have fresh outputs from Blocks 2, 7, and 14.
# Change selected_time_frames in Block 4 to a list like [21, 50, 200], then rerun the notebook.

def benchmark_zscore_for_plot(series):
    clean = pd.Series(series).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    zscore_series = calculate_zscore(clean)
    if zscore_series.isna().all():
        return pd.Series(0.0, index=clean.index)
    return zscore_series.dropna()

benchmark_time_frame_map = selected_time_frame_map if selected_time_frames is not None else risk_time_frame_map
benchmark_term_order = list(benchmark_time_frame_map)

if benchmark_order:
    asset_zscore_map = {
        term: benchmark_zscore_for_plot(asset_sharpe_map[term])
        for term in benchmark_term_order
        if term in asset_sharpe_map
    }

    summary_zscore_map = {}
    for term in benchmark_term_order:
        term_series_map = spread_plot_data.get(term, {})
        summary_zscore_map[term] = {}
        for symbol in benchmark_order:
            summary_zscore_map[term][symbol] = benchmark_zscore_for_plot(
                term_series_map.get(symbol, pd.Series(dtype=float))
            )

    detail_zscore_map = {}
    for symbol in benchmark_order:
        detail_zscore_map[symbol] = {}
        symbol_metrics = benchmark_metrics.get(symbol, {})
        for term in benchmark_term_order:
            term_metrics = symbol_metrics.get(term, {})
            asset_components = asset_component_map.get(term, {})
            detail_zscore_map[symbol][term] = {
                "asset": asset_zscore_map.get(term, pd.Series(dtype=float)),
                "benchmark": benchmark_zscore_for_plot(term_metrics.get("sharpe_ratio", pd.Series(dtype=float))),
                "asset_sharpe": asset_sharpe_map.get(term, pd.Series(dtype=float)).dropna(),
                "benchmark_sharpe": term_metrics.get("sharpe_ratio", pd.Series(dtype=float)).dropna(),
                "asset_excess_return": asset_components.get("annualized_excess_return", pd.Series(dtype=float)).dropna(),
                "benchmark_excess_return": term_metrics.get("annualized_excess_return", pd.Series(dtype=float)).dropna(),
                "asset_volatility": asset_components.get("annualized_volatility", pd.Series(dtype=float)).dropna(),
                "benchmark_volatility": term_metrics.get("annualized_volatility", pd.Series(dtype=float)).dropna(),
                "sharpe_spread": benchmark_zscore_for_plot(term_metrics.get("sharpe_spread", pd.Series(dtype=float))),
                "relative_spread": benchmark_zscore_for_plot(term_metrics.get("spread", pd.Series(dtype=float))),
            }

    summary_fig = plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=summary_zscore_map,
        time_frame_map=benchmark_time_frame_map,
        ticker_label=ticker_str,
    )
    summary_fig.show()

    detail_fig = plot_benchmark_zscore_detail(
        detail_zscore_map=detail_zscore_map,
        benchmark_order=benchmark_order,
        time_frame_map=benchmark_time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=default_benchmark,
    )
    detail_fig.show()
else:
    print("No benchmark data available for benchmark comparison plots.")
